# 13 · Clip polygons against shapely

**Question.** Does Fieldwork's `clip_polygons` widget intersect polygons with a boundary the
way GEOS does, and does it drop the ones that fall outside entirely?

Fieldwork intersects with **polyclip-ts**; the reference intersects with **GEOS**. Both work
on longitude and latitude directly, because that is what the widget declares — it does not
project first. The check compares what the widget claims, not what it might have claimed.

The fixture is the John Snow example's own chain: nearest-pump Voronoi cells clipped to the
reporting rectangle. The cells are computed in a finite envelope and extend past the
rectangle, so the clip is doing real work — an unclipped cell claims ground the report does
not cover.

Criteria fixed before running: the same polygons surviving, each within 1e-13 square degrees
of shapely's intersection by symmetric difference, and no output polygon extending beyond the
boundary. Tolerances are in square degrees because the operation is; at this latitude 1e-13
square degrees is under a square millimetre on the ground.

In [ ]:
# Only what this notebook imports. Requesting rasterio alongside pyproj kills this kernel
# outright; see 00-kernel-check for the measured map.
NEEDED = ["shapely", "pyproj", "numpy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

## Parity

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_13_clip_polygons import run

report = run(Path.cwd() / "fixtures" / "clip-polygons-soho-001.json")
print("criterion:", report["criterion"])
print()
print("agrees:", report["agrees"])
print(f"polygons in            {report['polygonsIn']}")
print(f"survived, fieldwork    {report['polygonsOut']}")
print(f"survived, shapely      {report['referenceOut']}")
print(f"worst symmetric difference  {report['worstSymmetricDifferenceSquareDegrees']:.3e} square degrees")
print(f"polygons escaping the boundary  {report['escapedTheBoundary'] or 'none'}")
print()
for row in report["perPolygon"]:
    print(f"  {row['id']:<12} {row['symmetricDifferenceSquareDegrees']:.3e}")

## Dive deeper · the assumption is that degrees will do

The widget intersects on longitude and latitude without projecting. That is a declared
simplification, not an oversight, and the question a practitioner should be able to answer is
**when it stops being safe**.

An intersection is a geometric predicate, so the answer is not "it is wrong by the cosine of
the latitude" — clipping a polygon by a rectangle gives the same *answer* in either space
when the edges are straight in both. It stops being safe when an edge that is straight in
degrees is curved on the ground, or vice versa, and that depends on how long the edge is.

In [ ]:
import numpy as np, pyproj, shapely
from shapely.geometry import shape
from shapely.ops import transform

fixture = json.loads((Path.cwd() / "fixtures" / "clip-polygons-soho-001.json").read_text())
boundary = shape(fixture["input"]["boundary"])
to_metric = pyproj.Transformer.from_crs("OGC:CRS84", "EPSG:32630", always_xy=True)
to_degrees = pyproj.Transformer.from_crs("EPSG:32630", "OGC:CRS84", always_xy=True)
project = lambda geometry, t: shapely.transform(
    geometry, lambda coords: np.column_stack(t.transform(coords[:, 0], coords[:, 1])))

west, south, east, north = boundary.bounds
print(f"the reporting rectangle spans {east - west:.3f} deg longitude by {north - south:.3f} deg latitude")
print()
print(f"{'cell':<12} {'in degrees, m2':>16} {'in metres, m2':>16} {'difference':>12}")
worst = 0.0
for polygon in fixture["input"]["polygons"]:
    cell = shape(polygon["geometry"])
    in_degrees = project(cell.intersection(boundary), to_metric).area
    in_metres = project(cell, to_metric).intersection(project(boundary, to_metric)).area
    if in_degrees == 0 and in_metres == 0:
        continue
    relative = abs(in_degrees - in_metres) / max(in_degrees, in_metres)
    worst = max(worst, relative)
    print(f"{polygon['id']:<12} {in_degrees:16.1f} {in_metres:16.1f} {relative:11.2e}")
print()
print(f"worst relative difference {worst:.2e} -- clipping in degrees and clipping in metres")
print("agree to that, over a rectangle this size. The simplification is safe here.")

### Where it would not be

The rectangle above is about 1.4 km across. The difference between clipping in degrees and
clipping in metres grows with the length of the edges being intersected, because a straight
line in one space bends in the other. For a boundary spanning degrees rather than
thousandths of one — a province, a coastline, a country — the two answers diverge visibly,
and the widget's note says so.

What that means in practice: this widget is safe for the reporting extents a field workflow
uses and should not be trusted for a continental clip without being reprojected first.

### Reading the result

`agrees` means polyclip-ts and GEOS intersected the same polygons to the same ground. It
does not mean clipping to a reporting rectangle is the right thing to do: a clipped catchment
is the part of that catchment the report covers, which is a different object from the
catchment, and both implementations produce it identically.